# Generate processed datasets

This notebook turns the base 5-minute dataset into regime, winsorized, and filtered variants for model comparison.


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "Data").exists() and (candidate / "Notebooks").exists():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
INPUT = DATA_DIR / "train_features_5m_clean.parquet"
OUTDIR = DATA_DIR / "processed_sets"

print("INPUT =", INPUT)
print("OUTDIR =", OUTDIR)


In [ ]:
def load_df(path=INPUT):
    df = pd.read_parquet(path)
    if "bar5" in df.columns:
        df["bar5"] = pd.to_datetime(df["bar5"])
    return df

# quick load test
df = load_df()
print(df.shape)


In [ ]:
def remove_april_peak(df, date_col="bar5"):
    start = pd.Timestamp("2025-04-02")
    end = pd.Timestamp("2025-04-10")
    mask = (df[date_col] < start) | (df[date_col] > end)
    df_removed = df[mask].copy()
    return df_removed, start.date(), end.date()


ds1, removed_start, removed_end = remove_april_peak(df)
print(removed_start, removed_end, ds1.shape)


In [ ]:
def winsorize_df(df, col="target_ann_vol", lower_q=0.01, upper_q=0.99):
    lower = float(df[col].quantile(lower_q))
    upper = float(df[col].quantile(upper_q))
    df_w = df.copy()
    df_w[col] = df_w[col].clip(lower=lower, upper=upper)
    return df_w, {col: lower}, {col: upper}


df_w, lq, uq = winsorize_df(df)
print(df_w.shape)


In [ ]:
def split_regimes(df, target_col="target_ann_vol", date_col="bar5"):
    daily_target = df.groupby(df[date_col].dt.floor("D"))[target_col].mean()
    thr = float(daily_target.median())
    high_days = daily_target[daily_target > thr].index
    is_high_day = df[date_col].dt.floor("D").isin(high_days)
    high = df[is_high_day].copy()
    low = df[~is_high_day].copy()
    return high, low, thr


high, low, thr = split_regimes(df)
print(thr, high.shape, low.shape)


In [ ]:
def summarize_and_save(df, name):
    OUTDIR.mkdir(exist_ok=True, parents=True)
    p = OUTDIR / f"{name}.parquet"
    df.to_parquet(p)
    return str(p)


p1 = summarize_and_save(ds1, "removed_peak_month")
p2 = summarize_and_save(df_w, "winsorized_1_99")
p_high = summarize_and_save(high, "regime_high_vol")
p_low = summarize_and_save(low, "regime_low_vol")
print(p1)
print(p2)
print(p_high)
print(p_low)


In [ ]:
summary = {
    "original_rows": int(len(df)),
    "original_cols": int(df.shape[1]),
    "removed_peak_month": {"rows": int(len(ds1)), "path": p1},
    "winsorized_1_99": {"rows": int(len(df_w)), "path": p2},
    "regime_threshold_target_ann_vol": thr,
    "regime_high_vol": {"rows": int(len(high)), "path": p_high},
    "regime_low_vol": {"rows": int(len(low)), "path": p_low},
}

with open(OUTDIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(summary)


In [ ]:
# Quick sanity check on the dataset splits
start = pd.Timestamp("2025-04-02")
end = pd.Timestamp("2025-04-10")
mask_removed_peak = (df["bar5"] >= start) & (df["bar5"] <= end)
q1 = df["target_ann_vol"].quantile(0.01)
q99 = df["target_ann_vol"].quantile(0.99)
mask_winsorized = (df["target_ann_vol"] < q1) | (df["target_ann_vol"] > q99)

removed_peak_rows = df[mask_removed_peak]
winsorized_rows = df[mask_winsorized]
overlap_rows = df[mask_removed_peak & mask_winsorized]

print("total rows:", len(df))
print("removed-peak rows:", len(removed_peak_rows))
print("winsorized extreme rows:", len(winsorized_rows))
print("overlap rows:", len(overlap_rows))
